# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 03: Graph Attention Networks (GAT)

### 1. Conceptual Intuition & Real-World Use Case
In standard GCNs, neighbor aggregation weights are static and depend solely on node degrees ($1/\sqrt{d_i d_j}$). All neighbors of equal degree are treated identically, regardless of their actual content or relevance.

In 2018, Veličković et al. introduced **Graph Attention Networks (GAT)**, porting the Transformer attention mechanism to arbitrary graphs.
- GAT assigns **dynamic, learnable attention weights** $\alpha_{ij}$ to each edge $(i, j)$.
- A paper can attend strongly to highly relevant cited papers while ignoring unrelated citations.
- Like Transformers, GAT uses **Multi-Head Attention** to stabilize learning and attend to different semantic relationships.

---

### 2. Mathematical Foundations
For node $i$ with feature $h_i$ and neighbor $j \in \mathcal{N}_i$:

1. **Shared Linear Transformation:**
   $$z_i = W h_i, \quad W \in \mathbb{R}^{F' \times F}$$

2. **Self-Attention Coefficients (Pairwise Compatibility):**
   $$e_{ij} = \text{LeakyReLU}\left( \mathbf{a}^T [z_i \parallel z_j] \right)$$
   where $\mathbf{a} \in \mathbb{R}^{2 F'}$ is a learnable attention parameter vector, and $\parallel$ denotes concatenation.

3. **Masked Softmax Normalization:**
   $$\alpha_{ij} = \text{Softmax}_j(e_{ij}) = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}_i \cup \{i\}} \exp(e_{ik})}$$
   Crucially, softmax is only computed over the 1-hop topological neighborhood $\mathcal{N}_i$. Non-adjacent nodes have zero attention weight.

4. **Multi-Head Aggregation ($K$ independent heads):**
   $$h_i^{(l+1)} = \Big|_{k=1}^K \sigma\left( \sum_{j \in \mathcal{N}_i} \alpha_{ij}^k W^k h_j \right)$$
   For the final classification layer, concatenation is replaced with averaging:
   $$h_i^{(final)} = \sigma\left( \frac{1}{K} \sum_{k=1}^K \sum_{j \in \mathcal{N}_i} \alpha_{ij}^k W^k h_j \right)$$

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Create a 5-node graph with features
N = 5
F_in = 4   # Input feature dimension
F_out = 3  # Output dimension per attention head
K_heads = 2 # Multi-head attention count

# Node features
X = np.random.randn(N, F_in)

# Adjacency list with self-loops
adj = {
    0: [0, 1, 2],
    1: [1, 0, 3],
    2: [2, 0, 3, 4],
    3: [3, 1, 2],
    4: [4, 2]
}

def leaky_relu(x, alpha=0.2):
    return np.where(x > 0, x, alpha * x)

# Step 2: Implement Single-Head Graph Attention Layer
class GATHead:
    def __init__(self, in_features, out_features):
        # Weight projection W: (F_in, F_out)
        self.W = np.random.randn(in_features, out_features) * 0.4
        # Attention vector a: (2 * F_out,)
        self.a = np.random.randn(2 * out_features) * 0.4

    def forward(self, X, adj):
        N = X.shape[0]
        # Step A: Linear transformation Z = X * W
        Z = X @ self.W  # (N, F_out)
        
        # Step B: Compute pairwise attention scores e_ij for existing edges
        alpha = np.zeros((N, N))
        
        for i in range(N):
            neighbors = adj[i]
            # Construct concatenated pairs [z_i || z_j]
            z_i = Z[i]  # (F_out,)
            scores = []
            for j in neighbors:
                z_j = Z[j]
                cat_ij = np.concatenate([z_i, z_j])  # (2 * F_out,)
                e_ij = leaky_relu(np.dot(self.a, cat_ij))
                scores.append(e_ij)
            
            # Step C: Softmax over neighbors
            scores = np.array(scores)
            exp_scores = np.exp(scores - np.max(scores))
            probs = exp_scores / np.sum(exp_scores)
            
            for idx, j in enumerate(neighbors):
                alpha[i, j] = probs[idx]
                
        # Step D: Aggregation H_new = Alpha * Z
        H_new = alpha @ Z
        return H_new, alpha

# Step 3: Run Multi-Head GAT
head1 = GATHead(F_in, F_out)
head2 = GATHead(F_in, F_out)

H1, alpha1 = head1.forward(X, adj)
H2, alpha2 = head2.forward(X, adj)

# Concatenate across heads: shape (N, K_heads * F_out) = (5, 6)
H_multi = np.concatenate([H1, H2], axis=1)

print("Input X shape:", X.shape)
print("Multi-Head Output shape (concatenated 2 heads):", H_multi.shape)

print("\nLearned Attention Weights (Head 1) for Node 2 neighbors:")
for j in adj[2]:
    print(f"  alpha(Node 2 -> Node {j}): {alpha1[2, j]:.4f}")
print("Sum of attention weights for Node 2:", np.sum(alpha1[2, :]))

### 3. Step-by-Step Code Walkthrough
1. **Linear Projection:** $X W$ transforms node features into an embedding space of dimension $F_{out}$.
2. **Pairwise Concatenation & LeakyReLU:** For every edge $(i, j)$, features $[z_i \parallel z_j]$ are inner-producted with attention vector $\mathbf{a}$.
3. **Masked Softmax:** Normalization is restricted strictly to topological neighbors ($j \in \mathcal{N}_i$). Unconnected nodes receive weight 0.
4. **Multi-Head Concatenation:** Outputs from independent heads are concatenated along the feature dimension, capturing distinct relational patterns (e.g. one head attends to co-authorship while another attends to topic citations).

---

### 4. Industry Applications & Trade-offs
- **Fraud Ring Detection:** GAT automatically learns to down-weight noisy random connections and focus attention on fraudulent accomplice edges.
- **Biochemical Reaction Modeling:** Predicting which atom bonds are reactive under specific conditions.
- **Trade-off:** Computing pairwise attention scores for all edges increases memory consumption compared to fixed-weight GCNs.

---

### 5. Top Interview Questions & Answers
**Q1: How does GAT differ from standard Transformer self-attention?**
> **Answer:** Standard Transformer attention is full $O(N^2)$ cross-attention across all token pairs without structural constraints. GAT is **masked self-attention** restricted solely to the 1-hop graph adjacency structure $\mathcal{N}_i$, scaling with $O(|E|)$ instead of $O(N^2)$.

**Q2: Why is LeakyReLU used with negative slope 0.2 before Softmax in GAT?**
> **Answer:** Standard ReLU zeroes out negative pre-attention logits, destroying gradient flow for lower-scoring neighbors. LeakyReLU allows small gradients ($alpha=0.2$) to backpropagate even for less important neighbors, ensuring stable weight updates during early training.